In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import torchaudio
from torchaudio.models.conformer import Conformer

import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import ast
import gc

from phoneme_encoder import phoneme_to_id

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
torch.cuda.empty_cache()
while gc.collect() > 0:
    print("cleaning ...........")

In [ ]:
CSV_PATH = "ayat_with_phonemes.csv"
BATCH_SIZE = 1
SR = 16000
EPOCHS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
bundle = torchaudio.pipelines.WAV2VEC2_BASE
wav2vec = bundle.get_model().to(DEVICE)
for param in wav2vec.parameters():
    param.requires_grad = True

In [ ]:
df = pd.read_csv(CSV_PATH)
df.info()

df = df.tail(1)

In [ ]:
def load_audio(path):
    wave, sr = torchaudio.load(path)
    if wave.shape[0] > 1:
        wave = torch.mean(wave, dim=0, keepdim=True)
    if sr != SR:
        resampler = torchaudio.transforms.Resample(sr, SR)
        wave = resampler(wave)

    max_val = torch.max(torch.abs(wave)) + 1e-8
    wave = wave / max_val

    return wave

In [ ]:
class QuranDs(Dataset):
    def __init__(self, data: pd.DataFrame):
        super().__init__()
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        labels, _ = ast.literal_eval(self.data.iloc[index]["phonemes"])
        labels = [phoneme_to_id[l] for l in labels]
        waveform = load_audio(self.data.iloc[index]["path_of_audio"])
        return waveform, torch.tensor(labels, dtype=torch.long)

In [ ]:
def collate_fn(batch):
    waves, labels = zip(*batch)
    waves = [wave.squeeze(0) for wave in waves]
    padded_waves = pad_sequence(waves, batch_first=True, padding_value=0)

    input_len = [w.size(0) for w in waves]

    target_len = [len(labels[i]) for i in range(len(labels))]

    return (
        padded_waves.to(DEVICE),
        torch.cat(labels, dim=0).to(DEVICE),
        torch.tensor(input_len, device=DEVICE),
        torch.tensor(target_len, device=DEVICE),
    )

In [ ]:
train_data = QuranDs(data=df)
train_dl = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)

In [ ]:
class QuranASRModel(torch.nn.Module):
    def __init__(self, feature_extractor, input_dim, hidden_size, num_classes):
        super().__init__()

        self.feature_extractor = feature_extractor

        self.conformer = Conformer(
            input_dim=input_dim,
            num_heads=256,
            ffn_dim=hidden_size,
            num_layers=6,
            depthwise_conv_kernel_size=31,
        )

        self.classifier = nn.Linear(
            in_features= input_dim , out_features=input_dim*4
        )
        
        self.classifier1 = nn.Linear(
            in_features= input_dim*4 , out_features=num_classes
        )

    def forward(self, wave, input_len):

        self.feature_extractor.eval()
        with torch.no_grad():
            features, features_len = self.feature_extractor(wave, input_len)

        conformer_out, features_len = self.conformer(features,features_len)

        logits = self.classifier(conformer_out)
        logits = self.classifier1(logits)


        return logits, features_len

In [ ]:
model = QuranASRModel(
    feature_extractor=wav2vec,
    hidden_size=512,
    input_dim=768,
    num_classes=len(phoneme_to_id),
).to(DEVICE)

In [ ]:
wave, label, input_len, target_len = next(iter(train_dl))

print("wave shape :", wave.shape)
print("label shape :", label.shape) 
print("input length :", input_len)
print("target length :", target_len)

logits, features_len = model(wave, input_len)

print("features_len shape :", features_len)


preds = logits.argmax(dim=2)

print("preds shape", preds.shape)

In [ ]:
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

ctc_loss = torch.nn.CTCLoss()


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=4,min_lr=0.00001
)

In [ ]:
best_loss=float("inf")

no_improvement_epochs = 0

patience_threshold = 10

loss_plot = []

for epoch in range(EPOCHS):

    train_loss = 0

    for wave, label, input_len, target_len in tqdm(train_dl):

        optimizer.zero_grad()

        logits, features_len = model(wave, input_len)

        log_prob = logits.log_softmax(dim=2)

        loss = ctc_loss(
            log_prob.transpose(0, 1),
            label,
            features_len,
            target_len,
        )

        loss.backward()
        optimizer.step()


        train_loss += loss.item()


    train_loss /= len(train_dl)

    loss_plot.append(train_loss)

    if train_loss < best_loss:
        best_loss = train_loss
        no_improvement_epochs = 0
    else:
        no_improvement_epochs += 1

    if no_improvement_epochs >= patience_threshold:
        print("Early stopping triggered.")
        break

    scheduler.step(train_loss)


    print("epoch : ", epoch + 1)

    print("train_loss: {:.4f}".format(train_loss))
    print("learning rate: {:.6f}".format(optimizer.param_groups[0]["lr"]))

    print("-" * 50)

In [ ]:
plt.plot(loss_plot)
plt.xlabel("Epochs")
plt.ylabel("Training Loss")
plt.title("Training Loss over Epochs")
plt.grid()
plt.show()

In [ ]:
# def ctc_decode(predicted_text):

#     text = " "

#     for char in predicted_text:
#         if char != text[-1] and char != BLANK_TOKEN:
#             text += char

#     return text.strip()